# EDA — `INSYTE_TRAKING_EVENTS`

Exploratory analysis of the tracking events table behind the Analytics Dashboard.

**How this is built.** Every aggregation runs in Snowflake and only small result
frames come back to pandas — the table is millions of rows, so `SELECT *` into a
DataFrame is not an option. Charts use Altair.

**Packages.** Nothing beyond `snowflake-snowpark-python`, `pandas`, and `altair`,
all of which are already present on both the warehouse and container runtimes. No
External Access Integration needed.

**Structure.** Sections 1–3 and 11 discover the schema and profile whatever columns
actually exist, so they work unchanged if the table differs from expectations.
Sections 4–10 target specific columns (`EVENT_TS`, `SESSION_ID`, `EVENT_NAME`,
`EVENT_TYPE`, `REQUEST_IP`, `CAMPAIGN_ID`, `CHANNEL`, `PROPERTIES`) — if section 1
shows different names, adjust those cells. Each is independent, so one failing
does not block the rest.

In [ ]:
import altair as alt
import pandas as pd

# Snowflake Notebooks need the mimetype renderer to display Vega content; without
# this, chart cells fail with "Call alt.renderers.enable('mimetype')".
alt.renderers.enable("mimetype")
alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

TABLE = "CIT_DATA_CORE.TRACKING.INSYTE_TRAKING_EVENTS"
DB, SCHEMA, TABLE_NAME = TABLE.split(".")

BLUE = "#2a78d6"
ORANGE = "#eb6834"


def _connect():
    """Active session inside a Snowflake Notebook; local secrets.toml otherwise."""
    try:
        from snowflake.snowpark.context import get_active_session
        return get_active_session()
    except Exception:
        pass
    import tomllib
    from pathlib import Path
    from snowflake.snowpark import Session
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".streamlit" / "secrets.toml"
        if not candidate.exists():
            continue
        cfg = tomllib.loads(candidate.read_text(encoding="utf-8"))["connections"]["snowflake"]
        return Session.builder.configs(cfg).create()
    raise RuntimeError("No active Snowflake session, and no .streamlit/secrets.toml found")


session = _connect()


def q(sql):
    """Run SQL and return a DataFrame. Aggregate server-side - never fetch raw rows."""
    return session.sql(sql).to_pandas()


q("SELECT CURRENT_ROLE() AS ROLE, CURRENT_WAREHOUSE() AS WAREHOUSE, CURRENT_VERSION() AS VERSION")

## 1. Schema

What columns actually exist, and their types. Everything downstream keys off this.

In [ ]:
COLS = q(f"""
SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE, CHARACTER_MAXIMUM_LENGTH
FROM {DB}.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = '{SCHEMA}' AND TABLE_NAME = '{TABLE_NAME}'
ORDER BY ORDINAL_POSITION
""")

COLUMN_NAMES = COLS["COLUMN_NAME"].tolist()
COLUMN_TYPES = dict(zip(COLS["COLUMN_NAME"], COLS["DATA_TYPE"]))
print(f"{len(COLUMN_NAMES)} columns")
COLS

## 2. Scale and time coverage

Row count and date span — reported twice, deliberately.

Raw `MIN`/`MAX` on `EVENT_TS` are worthless here: a couple of hundred corrupt rows
out of ~22 million stretch the apparent range across decades. The *plausible* bounds
ignore anything before 2020 or in the future, and those are the numbers to trust.

**Compare the plausible range against the dashboard's default 30-day window.** If the
data ended well before today, every page would render empty while working correctly.

In [ ]:
scale = q(f"""
SELECT
    COUNT(*) AS ROW_COUNT,
    MIN(EVENT_TS) AS RAW_MIN,
    MAX(EVENT_TS) AS RAW_MAX,
    SUM(IFF(EVENT_TS > CURRENT_TIMESTAMP(), 1, 0)) AS FUTURE_ROWS,
    SUM(IFF(EVENT_TS < DATE '2020-01-01', 1, 0)) AS PRE_2020_ROWS,
    MIN(IFF(EVENT_TS BETWEEN DATE '2020-01-01' AND CURRENT_TIMESTAMP(), EVENT_TS, NULL)) AS PLAUSIBLE_MIN,
    MAX(IFF(EVENT_TS BETWEEN DATE '2020-01-01' AND CURRENT_TIMESTAMP(), EVENT_TS, NULL)) AS PLAUSIBLE_MAX
FROM {TABLE}
""")

TOTAL_ROWS = int(scale["ROW_COUNT"].iloc[0])
outliers = int(scale["FUTURE_ROWS"].iloc[0]) + int(scale["PRE_2020_ROWS"].iloc[0])
print(f"{TOTAL_ROWS:,} rows")
print(f"raw range        {scale['RAW_MIN'].iloc[0]}  ->  {scale['RAW_MAX'].iloc[0]}")
print(f"plausible range  {scale['PLAUSIBLE_MIN'].iloc[0]}  ->  {scale['PLAUSIBLE_MAX'].iloc[0]}")
print(f"{outliers} rows have out-of-range timestamps and are what distort the raw range")
scale.T

In [ ]:
recent = q(f"""
SELECT COUNT(*) AS EVENTS, COUNT(DISTINCT SESSION_ID) AS SESSIONS, COUNT(DISTINCT CAMPAIGN_ID) AS CAMPAIGNS
FROM {TABLE}
WHERE EVENT_TS::DATE BETWEEN DATEADD(day, -30, CURRENT_DATE()) AND CURRENT_DATE()
""")

print("Exactly what the dashboard's default filter selects:")
recent

## 3. Completeness

Non-null count and null percentage for every column, in one pass. A column that is
mostly null cannot carry a KPI — this is where you find that out before designing
one around it.

In [ ]:
count_exprs = ", ".join([f'COUNT("{c}") AS "{c}"' for c in COLUMN_NAMES])
raw = q(f"SELECT {count_exprs} FROM {TABLE}")

completeness = raw.T.rename(columns={0: "NON_NULL"})
completeness["NULL_PCT"] = (1 - completeness["NON_NULL"] / TOTAL_ROWS) * 100
completeness["DATA_TYPE"] = [COLUMN_TYPES.get(i, "") for i in completeness.index]
completeness.sort_values("NULL_PCT", ascending=False)

In [ ]:
plot_df = completeness.reset_index(names="COLUMN").sort_values("NULL_PCT", ascending=False)

alt.Chart(plot_df).mark_bar(color=ORANGE).encode(
    x=alt.X("NULL_PCT:Q", title="Null %", scale=alt.Scale(domain=[0, 100])),
    y=alt.Y("COLUMN:N", title=None, sort="-x"),
    tooltip=["COLUMN", alt.Tooltip("NULL_PCT:Q", format=".1f"), alt.Tooltip("NON_NULL:Q", format=",")],
).properties(height=alt.Step(18))

## 4. Cardinality

Approximate distinct count per column. This separates dimensions you can group by
(low cardinality) from identifiers (near-unique). Semi-structured columns are
excluded here and explored in section 10.

In [ ]:
SEMI_STRUCTURED = {"VARIANT", "OBJECT", "ARRAY"}
plain_cols = [c for c in COLUMN_NAMES if COLUMN_TYPES.get(c, "").upper() not in SEMI_STRUCTURED]

approx_exprs = ", ".join([f'APPROX_COUNT_DISTINCT("{c}") AS "{c}"' for c in plain_cols])
card = q(f"SELECT {approx_exprs} FROM {TABLE}").T.rename(columns={0: "APPROX_DISTINCT"})
card["PCT_OF_ROWS"] = card["APPROX_DISTINCT"] / TOTAL_ROWS * 100
card.sort_values("APPROX_DISTINCT", ascending=False)

## 5. Volume over time

Daily events and sessions on independent axes. Look for gaps (collection outages),
step changes (tracking deployments), and whether the two series move together.

In [ ]:
daily = q(f"""
SELECT
    EVENT_TS::DATE AS EVENT_DATE,
    COUNT(*) AS EVENTS,
    COUNT(DISTINCT SESSION_ID) AS SESSIONS
FROM {TABLE}
GROUP BY 1
ORDER BY 1
""")

# Snowflake returns DATE as Python date objects, which Altair cannot serialise to
# JSON - charts then fail to render. Convert to datetime64 before plotting.
daily["EVENT_DATE"] = pd.to_datetime(daily["EVENT_DATE"])

base = alt.Chart(daily).encode(x=alt.X("EVENT_DATE:T", title=None))
events_line = base.mark_line(color=BLUE, strokeWidth=2).encode(y=alt.Y("EVENTS:Q", title="Events"))
sessions_line = base.mark_line(color=ORANGE, strokeWidth=2).encode(y=alt.Y("SESSIONS:Q", title="Sessions"))

alt.layer(events_line, sessions_line).resolve_scale(y="independent").properties(height=280)

In [ ]:
gaps = daily.copy()
gaps["PREV"] = gaps["EVENT_DATE"].shift(1)
gaps["GAP_DAYS"] = (gaps["EVENT_DATE"] - gaps["PREV"]).dt.days
missing = gaps[gaps["GAP_DAYS"] > 1]

print(f"{len(daily)} days with data")
print(f"{len(missing)} gaps of more than one day")
missing[["PREV", "EVENT_DATE", "GAP_DAYS"]]

## 6. Activity by weekday and hour

Human traffic clusters into business hours; flat or nocturnal activity points at
automation. Hours are in whatever timezone `EVENT_TS` is stored in — check that
before reading too much into the shape.

In [ ]:
heat = q(f"""
SELECT
    DAYNAME(EVENT_TS) AS WEEKDAY,
    HOUR(EVENT_TS) AS HOUR_OF_DAY,
    COUNT(*) AS EVENTS
FROM {TABLE}
GROUP BY 1, 2
""")

DAY_ORDER = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

alt.Chart(heat).mark_rect().encode(
    x=alt.X("HOUR_OF_DAY:O", title="Hour of day"),
    y=alt.Y("WEEKDAY:O", title=None, sort=DAY_ORDER),
    color=alt.Color("EVENTS:Q", title="Events", scale=alt.Scale(scheme="blues")),
    tooltip=["WEEKDAY", "HOUR_OF_DAY", alt.Tooltip("EVENTS:Q", format=",")],
).properties(height=220)

## 7. Events

How `EVENT_NAME` and `EVENT_TYPE` relate, and how concentrated the distribution is.
A handful of names usually dominate; the tail matters for whether "Top 10" is a
fair summary.

In [ ]:
events = q(f"""
SELECT EVENT_TYPE, EVENT_NAME, COUNT(*) AS EVENTS
FROM {TABLE}
GROUP BY 1, 2
ORDER BY EVENTS DESC
""")

events["SHARE_PCT"] = events["EVENTS"] / TOTAL_ROWS * 100
events["CUMULATIVE_PCT"] = events["SHARE_PCT"].cumsum()
print(f"{len(events)} distinct (type, name) pairs")
print(f"top 10 pairs cover {events['SHARE_PCT'].head(10).sum():.1f}% of all events")
events.head(25)

In [ ]:
top_names = events.groupby("EVENT_NAME", as_index=False)["EVENTS"].sum().nlargest(20, "EVENTS")

alt.Chart(top_names).mark_bar(color=BLUE).encode(
    x=alt.X("EVENTS:Q", title="Events"),
    y=alt.Y("EVENT_NAME:N", title=None, sort="-x"),
    tooltip=["EVENT_NAME", alt.Tooltip("EVENTS:Q", format=",")],
).properties(height=alt.Step(20))

## 8. Sessions

Session-level shape: events per session and duration. Expect heavy right skew —
the mean will sit well above the median, which is why the dashboard clips its
histograms to the 95th percentile.

In [ ]:
session_stats = q(f"""
WITH s AS (
    SELECT
        SESSION_ID,
        COUNT(*) AS EVENTS,
        DATEDIFF('second', MIN(EVENT_TS), MAX(EVENT_TS)) AS DURATION_S,
        COUNT(DISTINCT EVENT_TS::DATE) AS DISTINCT_DAYS
    FROM {TABLE}
    GROUP BY SESSION_ID
)
SELECT
    COUNT(*) AS SESSIONS,
    AVG(EVENTS) AS MEAN_EVENTS,
    APPROX_PERCENTILE(EVENTS, 0.5) AS P50_EVENTS,
    APPROX_PERCENTILE(EVENTS, 0.95) AS P95_EVENTS,
    MAX(EVENTS) AS MAX_EVENTS,
    AVG(DURATION_S) / 60 AS MEAN_DURATION_MIN,
    APPROX_PERCENTILE(DURATION_S, 0.5) / 60 AS P50_DURATION_MIN,
    APPROX_PERCENTILE(DURATION_S, 0.95) / 60 AS P95_DURATION_MIN,
    MAX(DURATION_S) / 60 AS MAX_DURATION_MIN,
    SUM(CASE WHEN EVENTS = 1 THEN 1 ELSE 0 END) AS SINGLE_EVENT_SESSIONS,
    SUM(CASE WHEN DURATION_S = 0 THEN 1 ELSE 0 END) AS ZERO_DURATION_SESSIONS,
    SUM(CASE WHEN DISTINCT_DAYS > 1 THEN 1 ELSE 0 END) AS MULTI_DAY_SESSIONS
FROM s
""")

session_stats.T

In [ ]:
session_hist = q(f"""
WITH s AS (SELECT SESSION_ID, COUNT(*) AS EVENTS FROM {TABLE} GROUP BY SESSION_ID)
SELECT LEAST(EVENTS, 40) AS EVENTS_BUCKET, COUNT(*) AS SESSIONS
FROM s
GROUP BY 1
ORDER BY 1
""")

alt.Chart(session_hist).mark_bar(color=BLUE).encode(
    x=alt.X("EVENTS_BUCKET:Q", title="Events per session (40 = 40 or more)"),
    y=alt.Y("SESSIONS:Q", title="Sessions"),
    tooltip=["EVENTS_BUCKET", alt.Tooltip("SESSIONS:Q", format=",")],
).properties(height=260)

## 9. Visitor proxy — `REQUEST_IP`

There is **no persistent visitor ID** in this table, so `REQUEST_IP` is the closest
stand-in. It is not identity: NAT, VPNs, and corporate networks collapse many people
into one address, and bots inflate it. The top-IPs table below is the check on how
badly that distorts any per-visitor metric.

In [ ]:
visitor_stats = q(f"""
WITH v AS (
    SELECT
        REQUEST_IP,
        COUNT(DISTINCT SESSION_ID) AS SESSIONS,
        COUNT(*) AS EVENTS,
        COUNT(DISTINCT EVENT_TS::DATE) AS ACTIVE_DAYS
    FROM {TABLE}
    GROUP BY REQUEST_IP
)
SELECT
    COUNT(*) AS DISTINCT_IPS,
    APPROX_PERCENTILE(SESSIONS, 0.5) AS P50_SESSIONS,
    APPROX_PERCENTILE(SESSIONS, 0.95) AS P95_SESSIONS,
    MAX(SESSIONS) AS MAX_SESSIONS,
    SUM(CASE WHEN ACTIVE_DAYS > 1 THEN 1 ELSE 0 END) AS IPS_ACTIVE_MULTIPLE_DAYS,
    SUM(CASE WHEN SESSIONS = 1 THEN 1 ELSE 0 END) AS SINGLE_SESSION_IPS
FROM v
""")

visitor_stats.T

In [ ]:
top_ips = q(f"""
SELECT
    REQUEST_IP,
    COUNT(DISTINCT SESSION_ID) AS SESSIONS,
    COUNT(*) AS EVENTS,
    COUNT(DISTINCT EVENT_TS::DATE) AS ACTIVE_DAYS
FROM {TABLE}
GROUP BY REQUEST_IP
ORDER BY SESSIONS DESC
LIMIT 15
""")

p95 = float(visitor_stats["P95_SESSIONS"].iloc[0])
top_ips["X_ABOVE_P95"] = top_ips["SESSIONS"] / p95
print(f"95th percentile is {p95:,.0f} sessions per IP")
top_ips

## 10. Campaigns, channels, and `PROPERTIES`

`CAMPAIGN_ID` is the stable key but is often an opaque UUID; a readable label lives
in `PROPERTIES:campaign_name`, which is only partly populated. The resolution rate
below decides whether campaign charts can be labelled honestly.

In [ ]:
campaign_stats = q(f"""
SELECT
    COUNT(DISTINCT CAMPAIGN_ID) AS DISTINCT_CAMPAIGN_IDS,
    COUNT(DISTINCT PROPERTIES:campaign_name::STRING) AS DISTINCT_CAMPAIGN_NAMES,
    COUNT(PROPERTIES:campaign_name::STRING) / COUNT(*) * 100 AS PCT_ROWS_WITH_NAME,
    COUNT(DISTINCT CHANNEL) AS DISTINCT_CHANNELS
FROM {TABLE}
""")

campaign_stats.T

In [ ]:
channels = q(f"""
SELECT COALESCE(CHANNEL, '(null)') AS CHANNEL, COUNT(*) AS EVENTS
FROM {TABLE}
GROUP BY 1
ORDER BY EVENTS DESC
""")

channels["SHARE_PCT"] = channels["EVENTS"] / TOTAL_ROWS * 100
channels

In [ ]:
property_keys = q(f"""
SELECT f.key AS PROPERTY_KEY, COUNT(*) AS OCCURRENCES
FROM (SELECT PROPERTIES FROM {TABLE} SAMPLE (200000 ROWS)) t,
     LATERAL FLATTEN(input => t.PROPERTIES) f
GROUP BY 1
ORDER BY OCCURRENCES DESC
""")

print("Keys found inside PROPERTIES (200k-row sample, so counts are indicative)")
property_keys

## 11. Data quality checks

Cheap assertions worth knowing before trusting any aggregate. Anything non-zero
here needs explaining, not ignoring.

In [ ]:
checks = q(f"""
SELECT
    SUM(IFF(SESSION_ID IS NULL, 1, 0)) AS NULL_SESSION_ID,
    SUM(IFF(EVENT_TS IS NULL, 1, 0)) AS NULL_EVENT_TS,
    SUM(IFF(EVENT_NAME IS NULL, 1, 0)) AS NULL_EVENT_NAME,
    SUM(IFF(EVENT_TS > CURRENT_TIMESTAMP(), 1, 0)) AS FUTURE_TIMESTAMPS,
    SUM(IFF(EVENT_TS < DATE '2020-01-01', 1, 0)) AS IMPLAUSIBLY_OLD,
    COUNT(*) - COUNT(DISTINCT MESSAGE_ID) AS NON_UNIQUE_MESSAGE_IDS
FROM {TABLE}
""")

flags = checks.T.rename(columns={0: "COUNT"})
flags["PCT_OF_ROWS"] = flags["COUNT"] / TOTAL_ROWS * 100
flags["CONCERN"] = flags["COUNT"] > 0
flags

`MESSAGE_ID` is used above as the candidate unique key. Don't substitute a synthetic
key like `SESSION_ID + EVENT_TS + EVENT_NAME` — many genuine events share a session
and timestamp to the second, so that reports enormous "duplicate" counts that mean
nothing.

Next: the same logical action recorded under several spellings. Any `EVENT_NAME`
appearing in more than one casing is silently split across separate bars in the Top
Events chart, understating each.

In [ ]:
casing = q(f"""
SELECT
    LOWER(EVENT_NAME) AS NORMALISED,
    COUNT(DISTINCT EVENT_NAME) AS SPELLINGS,
    ARRAY_AGG(DISTINCT EVENT_NAME) AS VARIANTS,
    COUNT(*) AS EVENTS
FROM {TABLE}
WHERE EVENT_NAME IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT EVENT_NAME) > 1
ORDER BY EVENTS DESC
""")

print(f"{len(casing)} event names appear under more than one spelling")
casing

## 12. Sample rows

Finally, look at actual records. Aggregates hide encoding quirks, sentinel values,
and stringified nulls that are obvious the moment you read a row.

In [ ]:
q(f"SELECT * FROM {TABLE} SAMPLE (10 ROWS)")

## What to take from this

Read the results in this order when deciding what the dashboard can support:

1. **Section 2** — does the data reach the present? Everything else is moot if not.
2. **Section 3** — which columns are complete enough to build a KPI on.
3. **Section 4** — which columns are dimensions and which are identifiers.
4. **Sections 8–9** — how skewed sessions and per-IP activity are, which decides
   whether means are honest or medians and percentiles are needed.
5. **Section 10** — whether campaigns can be labelled or only counted.
6. **Section 11** — anything non-zero to caveat.

For a metric that has to be exactly right, verify it against a direct SQL query
rather than reading it off a chart.